In [0]:
%sql

DESCRIBE bakery_pos_develop.bronze.pos_order_line

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW stg_pos_order_line AS SELECT
    id AS pos_order_line_id,
    order_id[0] AS pos_order_id,
    price_subtotal,
    price_subtotal_incl,
    price_unit,
    product_id[0] AS product_id,
    qty,
    total_cost,
    CAST(((price_unit * qty) * (discount / 100)) AS DECIMAL(10,2)) AS discount_amount,
    CAST((price_subtotal - total_cost) AS DECIMAL(10,2)) AS marging_amount,
    CAST(create_date AS TIMESTAMP) AS created_at,
    CAST(write_date AS TIMESTAMP) AS updated_at,
    processing_timestamp AS bronze_ingested_at
FROM bakery_pos_develop.bronze.pos_order_line
QUALIFY ROW_NUMBER() OVER(
    PARTITION BY id
    ORDER BY write_date DESC,
    processing_timestamp DESC
) = 1

In [0]:
%sql

CREATE TABLE IF NOT EXISTS bakery_pos_develop.silver.pos_order_line (
    pos_order_line_id BIGINT,
    pos_order_id BIGINT,
    price_subtotal DECIMAL(10, 2),
    price_subtotal_incl DECIMAL(10, 2),
    price_unit DECIMAL(10, 2),
    product_id BIGINT,
    qty DECIMAL(10, 2),
    total_cost DECIMAL(10, 2),
    discount_amount DECIMAL(10, 2),
    marging_amount DECIMAL(10, 2),
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    bronze_ingested_at TIMESTAMP,
    silver_processed_at TIMESTAMP,
    is_updated BOOLEAN,
    is_deleted BOOLEAN
) USING DELTA;

In [0]:
%sql

MERGE INTO bakery_pos_develop.silver.pos_order_line AS target
USING (
  SELECT *, current_timestamp() AS silver_processed_at 
  FROM stg_pos_order_line
) AS source
ON target.pos_order_line_id = source.pos_order_line_id
WHEN MATCHED AND source.updated_at > target.updated_at THEN
  UPDATE SET
    pos_order_id = source.pos_order_id,
    price_subtotal = source.price_subtotal,
    price_subtotal_incl = source.price_subtotal_incl,
    price_unit = source.price_unit,
    product_id = source.product_id,
    qty = source.qty,
    total_cost = source.total_cost,
    discount_amount = source.discount_amount,
    marging_amount = source.marging_amount,
    created_at = source.created_at,
    updated_at = source.updated_at,
    is_updated = True,
    is_deleted = False,
    bronze_ingested_at = source.bronze_ingested_at,
    silver_processed_at = source.silver_processed_at
WHEN NOT MATCHED THEN
  INSERT (
    pos_order_line_id,
    pos_order_id,
    price_subtotal,
    price_subtotal_incl,
    price_unit,
    product_id,
    qty,
    total_cost,
    discount_amount,
    marging_amount,
    created_at,
    updated_at,
    bronze_ingested_at,
    silver_processed_at,
    is_updated,
    is_deleted
  )
  VALUES (
    source.pos_order_line_id,
    source.pos_order_id,
    source.price_subtotal,
    source.price_subtotal_incl,
    source.price_unit,
    source.product_id,
    source.qty,
    source.total_cost,
    source.discount_amount,
    source.marging_amount,
    source.created_at,
    source.updated_at,
    source.bronze_ingested_at,
    source.silver_processed_at,
    False,
    False
  );